In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as pl

In [2]:
df = pd.read_csv('warfare_data.csv')
df.head()

,title,summary,content,links,url
0,War,NaN,NaN,['/wiki/Wikipedia:Protection_policy#semi'],https://en.wikipedia.org/wiki/War
1,Guerrilla warfare,Guerrilla warfare is a form of unconventional ...,Prehistoric\nAncient\nPost-classical\nCastles\...,"['/wiki/Swarming_(military)', '/wiki/Stay-behi...",https://en.wikipedia.org/wiki/Guerrilla_warfare
2,Trench warfare,\nTrench warfare is a type of land warfare us...,\nPrehistoric\nAncient\nPost-classical\nCastle...,"['/wiki/Dolomites', '/wiki/101st_Airborne_Divi...",https://en.wikipedia.org/wiki/Trench_warfare
3,The Ministry of Ungentlemanly Warfare,\nThe Ministry of Ungentlemanly Warfare is a 2...,\nPaul Tamasy\nEric Johnson\nArash Amel\nGuy R...,"['/wiki/Operation_Fortune:_Ruse_de_Guerre', '/...",https://en.wikipedia.org/wiki/The_Ministry_of_...
4,Nuclear warfare,"Nuclear warfare, also known as atomic warfare,...",Prehistoric\nAncient\nPost-classical\nCastles\...,"['/wiki/Conflict_escalation', '/wiki/Deforesta...",https://en.wikipedia.org/wiki/Nuclear_warfare


In [3]:
# as seen from the dataset above, we have to remove the \n from the summary and content column
# there are many \n in the content and the summary column
#this removes all the \n from the content table and summart
df["content"] = df["content"].str.replace("\n", " ", regex=True)
df["summary"] = df["summary"].str.replace("\n", " ", regex=True)
df.head()

,title,summary,content,links,url
0,War,NaN,NaN,['/wiki/Wikipedia:Protection_policy#semi'],https://en.wikipedia.org/wiki/War
1,Guerrilla warfare,Guerrilla warfare is a form of unconventional ...,Prehistoric Ancient Post-classical Castles Cas...,"['/wiki/Swarming_(military)', '/wiki/Stay-behi...",https://en.wikipedia.org/wiki/Guerrilla_warfare
2,Trench warfare,Trench warfare is a type of land warfare usi...,Prehistoric Ancient Post-classical Castles Ca...,"['/wiki/Dolomites', '/wiki/101st_Airborne_Divi...",https://en.wikipedia.org/wiki/Trench_warfare
3,The Ministry of Ungentlemanly Warfare,The Ministry of Ungentlemanly Warfare is a 20...,Paul Tamasy Eric Johnson Arash Amel Guy Ritch...,"['/wiki/Operation_Fortune:_Ruse_de_Guerre', '/...",https://en.wikipedia.org/wiki/The_Ministry_of_...
4,Nuclear warfare,"Nuclear warfare, also known as atomic warfare,...",Prehistoric Ancient Post-classical Castles Cas...,"['/wiki/Conflict_escalation', '/wiki/Deforesta...",https://en.wikipedia.org/wiki/Nuclear_warfare


In [4]:
df = df.drop(columns=['links'])
df.head()

,title,summary,content,url
0,War,NaN,NaN,https://en.wikipedia.org/wiki/War
1,Guerrilla warfare,Guerrilla warfare is a form of unconventional ...,Prehistoric Ancient Post-classical Castles Cas...,https://en.wikipedia.org/wiki/Guerrilla_warfare
2,Trench warfare,Trench warfare is a type of land warfare usi...,Prehistoric Ancient Post-classical Castles Ca...,https://en.wikipedia.org/wiki/Trench_warfare
3,The Ministry of Ungentlemanly Warfare,The Ministry of Ungentlemanly Warfare is a 20...,Paul Tamasy Eric Johnson Arash Amel Guy Ritch...,https://en.wikipedia.org/wiki/The_Ministry_of_...
4,Nuclear warfare,"Nuclear warfare, also known as atomic warfare,...",Prehistoric Ancient Post-classical Castles Cas...,https://en.wikipedia.org/wiki/Nuclear_warfare


In [6]:
#check on missing value
missing_values = df.isnull()
missing_values.head()
#list of columns with missing values
for column in missing_values.columns.values.tolist():
    print(column)
    print (missing_values[column].value_counts())
    print("")


title
title
False    9998
Name: count, dtype: int64

summary
summary
False    8629
True     1369
Name: count, dtype: int64

content
content
False    8636
True     1362
Name: count, dtype: int64

url
url
False    9998
Name: count, dtype: int64



In [30]:
df.duplicated().sum()
df = df.drop_duplicates()

In [31]:
df.to_csv('warfare_clean.csv')

In [32]:
from langdetect import detect
from deep_translator import GoogleTranslator


# Function to detect language
def detect_language(text):
    try:
        return detect(text)
    except:
        return "unknown"  # Handle errors

# Detect language
df['Language'] = df['content'].apply(detect_language)

# Filter only non-English rows
non_english_df = df[df['Language'] != 'en'].copy()  # Copy to avoid warnings

# Function to translate text
def translate_text(text):
    return GoogleTranslator(source='auto', target='en').translate(text)

# Apply translation **only to non-English rows**
non_english_df['EnglishText'] = non_english_df['Language'].apply(translate_text)

# Merge back translated texts into original DataFrame
df.update(non_english_df)

In [33]:
import wordninja
columns_to_fix = ['title', 'content', 'summary']
df[columns_to_fix] = df[columns_to_fix].astype(str).applymap(lambda x: " ".join(wordninja.split(x)))


C:\Users\lish\AppData\Local\Temp\ipykernel_9040\3573088578.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df[columns_to_fix] = df[columns_to_fix].astype(str).applymap(lambda x: " ".join(wordninja.split(x)))


In [34]:
df = df.reset_index(drop=True)

In [35]:
#remove urls
def remove_url(text):
    return re.sub(r'https?://\S+|www\.\S+', '', text)